# Aula 2 — Feature Engineering e EDA

Construção do dataset de modelagem (uma linha por partida, sem data leak) e análise exploratória das features.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px

from src.data import preparar_dados
from src.data.paths import PARTIDAS_PROC, DATASET_MODELAGEM
from src.features.build_features import construir_dataset, resumo_times

preparar_dados.executar()  # garante que os dados existem
ds = pd.read_csv(DATASET_MODELAGEM)
print('dataset:', ds.shape)
ds.head()

## Distribuição do alvo
O mandante vence com que frequência? Isso já é uma 'feature' implícita (mando de campo).

In [ ]:
ds['alvo'].value_counts(normalize=True).round(3)

## 1. Heatmap de correlação entre features

In [ ]:
mapa = {'Fora': -1, 'Empate': 0, 'Casa': 1}
num = ds.select_dtypes('number').copy()
num['alvo_num'] = ds['alvo'].map(mapa)
corr = num.corr(numeric_only=True).round(2)
px.imshow(corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto', height=700)

## 2. Boxplot: forma recente do mandante por resultado
Times que chegam em melhor forma tendem a vencer em casa?

In [ ]:
px.box(ds, x='alvo', y='casa_forma5', color='alvo',
       category_orders={'alvo': ['Casa', 'Empate', 'Fora']},
       labels={'casa_forma5': 'Forma do mandante (pts últimos 5)', 'alvo': 'Resultado'})

## 3. Scatter: aproveitamento em casa vs fora (por time)

In [ ]:
partidas = pd.read_csv(PARTIDAS_PROC)
r = resumo_times(partidas)
px.scatter(r, x='aprov_casa', y='aprov_fora', text='time', height=550,
           labels={'aprov_casa': 'Aproveitamento em casa', 'aprov_fora': 'Aproveitamento fora'})

## Reflexão
Quais features têm maior correlação com o resultado? Essas são candidatas fortes para o modelo da Fase 3.